In [ ]:
import os
import sys
import warnings

# Suppress Python deprecation warnings and TensorFlow background verbosity
warnings.filterwarnings("ignore")
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import tensorflow as tf

# Verify GPU is active
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"[OK] Hardware Accelerator detected: {gpus[0].name}")
else:
    print("[WARNING] No GPU detected. Make sure to set Runtime > Change runtime type > T4 GPU!")

[OK] Hardware Accelerator detected: /physical_device:GPU:0


In [ ]:
import os
import zipfile

# Your exact zip file path
ZIP_PATH = "/content/drive/MyDrive/dataset_validator.zip"
LOCAL_EXTRACT_DIR = "/content/dataset"

if not os.path.exists(LOCAL_EXTRACT_DIR):
    print(f"[INFO] Extracting {ZIP_PATH} to local disk. This will only take a few seconds...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(LOCAL_EXTRACT_DIR)
    print("[OK] Dataset extracted successfully to /content/dataset")
else:
    print("[INFO] Dataset already exists on local disk.")

[INFO] Dataset already exists on local disk.


In [ ]:
import os
print("Folders inside /content/dataset:", os.listdir("/content/dataset"))

Folders inside /content/dataset: ['dataset_validator']


In [ ]:
import os

# 1. Extract the .rar file using Colab's built-in unrar command
!unrar x /content/dataset/dataset_validator.rar /content/dataset/

# 2. Delete the .rar file so TensorFlow doesn't try to read it as an image
os.remove("/content/dataset/dataset_validator.rar")

# 3. Check what is inside the dataset folder now
print("Folders now inside /content/dataset:", os.listdir("/content/dataset"))


UNRAR 7.00 freeware      Copyright (c) 1993-2024 Alexander Roshal

Cannot open /content/dataset/dataset_validator.rar
No such file or directory
No files to extract


FileNotFoundError: [Errno 2] No such file or directory: '/content/dataset/dataset_validator.rar'

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# --- UPDATED PATH HERE ---
DATA_DIR = "/content/dataset/dataset_validator"
# -------------------------

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
PHASE_A_EPOCHS = 6
PHASE_B_EPOCHS = 4
MODEL_SAVE_PATH = "/content/kisanai_leaf_validator.keras"

print("[INFO] Loading datasets from local disk...")

# Load Training Split (80%)
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

# Load Validation Split (20%)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

class_names = train_ds.class_names
print(f"[OK] Detected classes: {class_names}")

# Optimize pipeline with in-memory caching and prefetching
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

# Data Augmentation Pipeline
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
], name="data_augmentation")

# Base Architecture (MobileNetV2 pretrained on ImageNet)
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False  # Freeze backbone for Phase A

# Model Assembly
inputs = tf.keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = tf.keras.Model(inputs, outputs, name="KisanAI_Leaf_Validator")

# Compile Model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# --- Phase A: Train Classification Head ---
print("\n[PHASE A] Training classification head (Backbone frozen)...")
history_phase_a = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=PHASE_A_EPOCHS,
    verbose=1
)

# --- Phase B: Fine-Tuning ---
print("\n[PHASE B] Fine-tuning upper layers...")
base_model.trainable = True

# Freeze lower layers, unfreeze top 30 layers
fine_tune_at = len(base_model.layers) - 30
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# Recompile with a lower learning rate to prevent catastrophic forgetting
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history_phase_b = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=PHASE_B_EPOCHS,
    verbose=1
)

# Save Model
model.save(MODEL_SAVE_PATH)
print(f"\n[SUCCESS] Validator model successfully saved at: {MODEL_SAVE_PATH}")

[INFO] Loading datasets from local disk...
Found 4192 files belonging to 2 classes.
Using 3354 files for training.
Found 4192 files belonging to 2 classes.
Using 838 files for validation.
[OK] Detected classes: ['leaf', 'non-leaf']

[PHASE A] Training classification head (Backbone frozen)...
Epoch 1/6
105/105 ━━━━━━━━━━━━━━━━━━━━ 15s 95ms/step - accuracy: 0.9636 - loss: 0.0851 - val_accuracy: 0.9940 - val_loss: 0.0220
Epoch 2/6
105/105 ━━━━━━━━━━━━━━━━━━━━ 7s 64ms/step - accuracy: 0.9919 - loss: 0.0248 - val_accuracy: 0.9952 - val_loss: 0.0116
Epoch 3/6
105/105 ━━━━━━━━━━━━━━━━━━━━ 7s 68ms/step - accuracy: 0.9890 - loss: 0.0295 - val_accuracy: 0.9952 - val_loss: 0.0138
Epoch 4/6
105/105 ━━━━━━━━━━━━━━━━━━━━ 7s 65ms/step - accuracy: 0.9946 - loss: 0.0170 - val_accuracy: 0.9952 - val_loss: 0.0124
Epoch 5/6
105/105 ━━━━━━━━━━━━━━━━━━━━ 7s 68ms/step - accuracy: 0.9869 - loss: 0.0341 - val_accuracy: 0.9916 - val_loss: 0.0195
Epoch 6/6
105/105 ━━━━━━━━━━━━━━━━━━━━ 8s 78ms/step - accuracy: 0.

In [ ]:
import shutil
shutil.copy("/content/kisanai_leaf_validator.keras", "/content/drive/MyDrive/kisanai_leaf_validator.keras")
print("[SUCCESS] Model copied to Google Drive!")

[SUCCESS] Model copied to Google Drive!


In [ ]:
import os
print("Files in /content/:", os.listdir("/content/"))
print("Files in /content/dataset/:", os.listdir("/content/dataset"))

Files in /content/: ['.config', 'kisanai_leaf_validator.keras', 'drive', 'dataset', 'sample_data']
Files in /content/dataset/: ['dataset_validator']


In [ ]:
DATA_DIR = "/content/dataset/dataset_validator"